# Alchemy

Solution author: Cowile

In [1]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from scipy.optimize import linear_sum_assignment

random.seed(2026)
np.random.seed(2026)
torch.manual_seed(2026)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
DATA = '/kaggle/input/competitions/alchemy-aicc-round-4' # replace this with your dataset path

train_df = pd.read_csv(f'{DATA}/train.csv')
test_df = pd.read_csv(f'{DATA}/test.csv')
candidates = sorted(pd.read_csv(f'{DATA}/candidates.csv')['result'].unique().tolist())

all_known_results = sorted(list(set(train_df['result'].unique()) | set(candidates)))

augmented = []
for _, row in train_df.iterrows():
    augmented.append({'item1': row['item1'], 'item2': row['item2'], 'result': row['result']})
    if row['item1'] != row['item2']:
        augmented.append({'item1': row['item2'], 'item2': row['item1'], 'result': row['result']})
train_aug = pd.DataFrame(augmented)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
template = 'combining {a} and {b} creates {r}'

In [3]:
class CrossEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for layer in self.bert.encoder.layer[:7]:
            for p in layer.parameters():
                p.requires_grad = False

        self.head = nn.Sequential(
            nn.Dropout(0.1), nn.Linear(768, 64), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(64, 1)
        )

    def score(self, texts):
        enc = tokenizer(texts, padding='max_length', truncation=True,
                        max_length=48, return_tensors='pt')
        ids = enc['input_ids'].to(device)
        mask = enc['attention_mask'].to(device)
        cls_emb = self.bert(ids, mask).last_hidden_state[:, 0]
        return self.head(cls_emb).squeeze(-1)

model = CrossEncoder().to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
backbone_params = [p for name, p in model.named_parameters()
                   if p.requires_grad and 'head' not in name]

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 2e-5},
    {'params': model.head.parameters(), 'lr': 1e-3}
])
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=30, num_training_steps=300)

for epoch in range(5):
    model.train()
    losses = []

    for _ in range(60):
        batch = train_aug.sample(16)
        items_a = batch['item1'].tolist()
        items_b = batch['item2'].tolist()
        correct_results = batch['result'].tolist()
        n = len(items_a)
        
        candidate_pool = list(set(correct_results))
        while len(candidate_pool) < n:
            distractor = random.choice(all_known_results)
            if distractor not in candidate_pool:
                candidate_pool.append(distractor)
        candidate_pool = candidate_pool[:n]
        random.shuffle(candidate_pool)

        target = torch.tensor([candidate_pool.index(r) for r in correct_results], device=device)

        texts = [template.format(a=items_a[i], b=items_b[i], r=candidate_pool[j])
                 for i in range(n) for j in range(n)]

        score_matrix = model.score(texts).view(n, n)
        loss = F.cross_entropy(score_matrix, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

    print(f'epoch {epoch + 1}  loss={np.mean(losses):.4f}')

epoch 1  loss=2.2215
epoch 2  loss=1.1922
epoch 3  loss=0.9005
epoch 4  loss=0.6759
epoch 5  loss=0.6293


In [5]:
model.eval()
score_mat = np.zeros((len(test_df), len(candidates)))

for idx, row in test_df.iterrows():
    i = test_df.index.get_loc(idx)

    forward_texts = [template.format(a=row['item1'], b=row['item2'], r=c) for c in candidates]
    reverse_texts = [template.format(a=row['item2'], b=row['item1'], r=c) for c in candidates]

    with torch.no_grad():
        fwd_scores = model.score(forward_texts)
        rev_scores = model.score(reverse_texts)

    score_mat[i] = (fwd_scores.cpu().numpy() + rev_scores.cpu().numpy()) / 2

_, col_assignments = linear_sum_assignment(-score_mat)
predictions = [candidates[c] for c in col_assignments]

In [6]:
submission = pd.DataFrame({'Id': test_df['Id'], 'result': predictions})
submission.to_csv('submission.csv', index=False)
print(f'saved {len(submission)} predictions')

saved 70 predictions
